In [7]:
import requests, zipfile, io
from datasets import Dataset, DatasetDict
from transformers import MarianTokenizer

url = "https://object.pouta.csc.fi/OPUS-KDE4/v2/moses/ar-en.txt.zip"
resp = requests.get(url)
resp.raise_for_status()

z = zipfile.ZipFile(io.BytesIO(resp.content))

with z.open("KDE4.ar-en.en") as f:
    en_lines = [line.decode("utf-8").strip() for line in f]
with z.open("KDE4.ar-en.ar") as f:
    ar_lines = [line.decode("utf-8").strip() for line in f]

valid_data = []
bad_patterns = ['@ info', '& #', '&#', 'Keyboard Layout']
for e, a in zip(en_lines, ar_lines):
    if len(e) > 0 and len(a) > 0:
        ratio = len(e) / len(a)
        if 0.25 <= ratio <= 4.0 and not any(p in e or p in a for p in bad_patterns):
            valid_data.append({"en": e, "ar": a})

raw_dataset = Dataset.from_dict({
    "id": [str(i) for i in range(len(valid_data))],
    "translation": valid_data
})

split_datasets = raw_dataset.train_test_split(train_size=0.9, seed=20)
split_datasets["validation"] = split_datasets.pop("test")
split_datasets


DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 93991
    })
    validation: Dataset({
        features: ['id', 'translation'],
        num_rows: 10444
    })
})

In [ ]:
sample = ["train"].shuffle(seed=44).select(range(5))

for row in sample:
    print(row)

{'id': '49462', 'translation': {'en': 'Click this button to save the current snapshot. To quickly save the snapshot without showing the file dialog, press Ctrl+Shift+S. The filename is automatically incremented after each save.', 'ar': 'اضغط هذا الزر لحفظ اللقطة الحاليّة. ولحفظ اللقطة بسرعة بدون عرض مربّع الحوار اضغط Ctrl+Shift+S. سيتزايد اسم الملف بعد كل عمليّة حفظ.'}}
{'id': '61407', 'translation': {'en': 'Desktop & resolution:', 'ar': '& دقة سطح المكتب:'}}
{'id': '37539', 'translation': {'en': 'These points are not collinear.', 'ar': 'هذه نقطة ليس.'}}
{'id': '42679', 'translation': {'en': 'Whether we use right-to-left typing.', 'ar': 'إذا ما ستستخدم الطباعة من اليمين- إلى- اليسار'}}
{'id': '88482', 'translation': {'en': 'Size Canvas to Size of Current Layer', 'ar': 'أعد تحجيم الصورة لحجم الطبقة الحالية'}}


In [11]:
from transformers import AutoModelForSeq2SeqLM, MarianTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-ar"

tokenizer = MarianTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [29]:
en_sentences = ["Windows is an OS", "I need to clear the cache and fix the bugs in the servers."]

inputs = tokenizer(en_sentences, return_tensors="pt", padding=True)

outputs = model.generate(**inputs)

predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print("-" * 30)
for en, ar in zip(en_sentences, predictions):
    print(f"English Input   : {en}")
    print(f"Model Prediction: {ar}")


------------------------------
English Input   : Windows is an OS
Model Prediction: النوافذ هي OSS
English Input   : I need to clear the cache and fix the bugs in the servers.
Model Prediction: أنا بحاجة لمسح المخبأ وإصلاح الحشرات في الخوادم.


In [ ]:
en_sentences = split_datasets["train"][0]["translation"]["en"]
ar_sentences = split_datasets["train"][0]["translation"]["ar"]
print(en_sentences, ar_sentences)


This link references the copyright. تشير هذه الوصلة إلى حقوق النسخ.


In [27]:
inputs = tokenizer(en_sentences, text_target=ar_sentences)
inputs

{'input_ids': [255, 6757, 14709, 3, 43677, 2, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1], 'labels': [1715, 71, 25786, 35, 27, 188, 20361, 2, 0]}

In [ ]:
max_length = 128
def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    target = [ex["ar"] for ex in examples["translation"]]

    model_inputs = tokenizer(inputs, text_target=target, max_length=max_length, truncation=True)
    return model_inputs



In [36]:
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names
    )


Map:   0%|          | 0/93991 [00:00<?, ? examples/s]

Map:   0%|          | 0/10444 [00:00<?, ? examples/s]

In [37]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [38]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [52]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[51226,     0, 62801, 62801, 62801, 62801, 62801, 62801, 62801, 62801],
        [ 1853,  8814,    14,  8940,   228,    14,   678, 26229, 15705,     0]]), 'attention_mask': tensor([[1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[23773,  6488, 32010,     0,  -100,  -100,  -100,  -100,  -100,  -100],
        [16404,  6571,    14,   256, 42636,    14, 14989, 12628,    93,     0]]), 'decoder_input_ids': tensor([[62801, 23773,  6488, 32010,     0, 62801, 62801, 62801, 62801, 62801],
        [62801, 16404,  6571,    14,   256, 42636,    14, 14989, 12628,    93]])})

In [54]:
batch["decoder_input_ids"]


tensor([[62801, 23773,  6488, 32010,     0, 62801, 62801, 62801, 62801, 62801],
        [62801, 16404,  6571,    14,   256, 42636,    14, 14989, 12628,    93]])

In [53]:
for i in range(1, 3):
    print(tokenized_datasets["train"][i]["labels"])

[23773, 6488, 32010, 0]
[16404, 6571, 14, 256, 42636, 14, 14989, 12628, 93, 0]


In [60]:
import numpy as np
import evaluate

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [57]:
from huggingface_hub import notebook_login

notebook_login()

In [65]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    f"marian-finetuned-kde4-en-to-ar",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True,
)

In [66]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [63]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Epoch,Bleu
No log,4.640252,0,7.449331


{'eval_loss': 4.640251636505127, 'eval_bleu': 7.449331124956469}

In [68]:
trainer.train()


Step,Training Loss
500,1.389108
1000,1.371088
1500,1.337991
2000,1.353291
2500,1.368835
3000,1.414533
3500,1.232472
4000,1.195688
4500,1.228640
5000,1.224149


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8814, training_loss=1.23910827151983, metrics={'train_runtime': 7782.0057, 'train_samples_per_second': 36.234, 'train_steps_per_second': 1.133, 'total_flos': 3815236343365632.0, 'train_loss': 1.23910827151983, 'epoch': 3.0})

In [69]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Step,Bleu
1.126744,1.267964,8814,37.875292


{'eval_loss': 1.2679640054702759, 'eval_bleu': 37.875291773038036}

In [70]:
trainer.push_to_hub(tags="translation", commit_message="Training complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/KhaledAalnobani/marian-finetuned-kde4-en-to-ar/commit/94bb24a0bf01e8e71a5a15de25b0f552d14b9224', commit_message='Training complete', commit_description='', oid='94bb24a0bf01e8e71a5a15de25b0f552d14b9224', pr_url=None, repo_url=RepoUrl('https://huggingface.co/KhaledAalnobani/marian-finetuned-kde4-en-to-ar', endpoint='https://huggingface.co', repo_type='model', repo_id='KhaledAalnobani/marian-finetuned-kde4-en-to-ar'), pr_revision=None, pr_num=None)

In [79]:
train_en = set(ex["en"] for ex in split_datasets["train"]["translation"])
val_en = set(ex["en"] for ex in split_datasets["validation"]["translation"])
overlap = train_en & val_en
print(f"Overlap: {len(overlap)} / {len(val_en)} ({100*len(overlap)/len(val_en):.1f}%)")

Overlap: 2305 / 9715 (23.7%)


In [ ]:
model_checkpoint ="KhaledAalnobani/marian-finetuned-kde4-en-to-ar" 
from transformers import AutoModelForSeq2SeqLM, MarianTokenizer


tokenizer = MarianTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
en_sentences = ["Windows is an OS", "I need to clear the cache and fix the bugs in the servers."]

inputs = tokenizer(en_sentences, return_tensors="pt", padding=True)

outputs = model.generate(**inputs)

predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print("-" * 30)
for en, ar in zip(en_sentences, predictions):
    print(f"English Input   : {en}")
    print(f"Model Prediction: {ar}")


tokenizer_config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/305M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

------------------------------
English Input   : Windows is an OS
Model Prediction: ويندوز هو نظام تشغيل
English Input   : I need to clear the cache and fix the bugs in the servers.
Model Prediction: يجب أن أمسح المخبأ وأصلح العلل في الخوادم.


 custom training loop

In [71]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets,
    collate_fn=data_collator,
    batch_size=8
)

In [72]:
model2 = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [73]:
from torch.optim import AdamW

optimizer = AdamW(model2.parameters(), lr=2e-5)


In [75]:
from accelerate import Accelerator

accelerator = Accelerator()
model2, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model2, optimizer, train_dataloader, eval_dataloader) 

In [76]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [77]:
def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels

In [78]:
from tqdm.auto import tqdm
import torch
from transformers import pipeline


output_dir ="marian-finetuned-kde4-en-to-ar-accelerate", 

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    model2.train()
    for batch in train_dataloader:
        outputs = model2(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    model2.eval()
    for batch in tqdm(eval_dataloader, desc=f"Evaluating Epoch {epoch}"):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model2).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length=128,
            )
        labels = batch["labels"]

        generated_tokens = accelerator.pad_across_processes(
            generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
        )
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)

        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    results = metric.compute()
    print(f"Epoch {epoch} | BLEU score: {results['score']:.2f}")

    accelerator.wait_for_everyone()
    unwrapped_model2 = accelerator.unwrap_model(model2)
    
    unwrapped_model2.save_pretrained(
        output_dir, 
        is_main_process=accelerator.is_main_process, 
        save_function=accelerator.save
    )
    
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        
        unwrapped_model2.push_to_hub(
            repo_id="KhaledAalnobani/marian-finetuned-kde4-en-to-ar-accelerate", 
            commit_message=f"Training in progress - Epoch {epoch}",
            safe_serialization=True 
        )
        tokenizer.push_to_hub(
            repo_id="KhaledAalnobani/marian-finetuned-kde4-en-to-ar-accelerate", 
            commit_message=f"Update tokenizer - Epoch {epoch}"
        )



  0%|          | 0/35247 [00:00<?, ?it/s]

KeyboardInterrupt: 